Section 1 — unknown 4,315건의 정체

Question: Are the unknown rows distributed across identities like a base set, and do they contradict the FSGAN hypothesis?

In [1]:
import os, sys
repo_root = os.getcwd()
if not os.path.exists(os.path.join(repo_root, 'src')):
    repo_root = os.path.abspath(os.path.join(repo_root, '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import csv, re, os
from collections import Counter
from src.report.style import setup, save
import matplotlib.pyplot as plt
setup()

rows=[]
index_path = os.path.join(repo_root, 'reports', 'eda', 'index.csv')
with open(index_path, newline='', encoding='utf-8') as f:
    rows=list(csv.DictReader(f))

unknown=[r for r in rows if r['method'].strip().lower()=='unknown']
faceswap=[r for r in rows if r['category']=='FakeVideo-RealAudio' and r['method'].strip().lower()=='faceswap']
wav=[r for r in rows if r['category']=='FakeVideo-RealAudio' and r['method'].strip().lower()=='wav2lip']
real=[r for r in rows if r['category']=='RealVideo-RealAudio']

# 1) identity coverage
unknown_id_counts=Counter(r['identity'] for r in unknown)
print('unknown identity count total', len(unknown_id_counts))
print('unknown identity counts summary', min(unknown_id_counts.values()), max(unknown_id_counts.values()), round(sum(unknown_id_counts.values())/len(unknown_id_counts), 2))
print('unknown top 20 counts')
for identity, count in unknown_id_counts.most_common(20):
    print(identity, count)

# 2) source ids in filenames
pattern=re.compile(r'id(\d{5})', re.IGNORECASE)
source_ids=Counter(m.group(1) for r in unknown if (m:=pattern.search(r['filename'])))
print('unknown source id distinct', len(source_ids))
print('unknown source id top 20')
for token, count in source_ids.most_common(20):
    print(token, count)

# 3) one identity's full file list
pick_identity='id05620'
pick_rows=[r for r in rows if r['identity']==pick_identity]
print('pick identity rows', len(pick_rows))
print('pick identity category/method counts')
for (category, method), count in Counter((r['category'], r['method']) for r in pick_rows).most_common():
    print(category, method, count)
print('pick identity filenames full list')
for r in pick_rows:
    print(r['filename'], '|', r['category'], '|', r['method'])

# 4) idless examples and compare with real filenames
idless=[r for r in unknown if not pattern.search(r['filename'])]
print('idless count', len(idless))
print('idless filenames 20', [r['filename'] for r in idless[:20]])
print('real filenames 20', [r['filename'] for r in real[:20]])

# 5) identity coverage compared to faceswap
ids_unknown=set(r['identity'] for r in unknown)
ids_faceswap=set(r['identity'] for r in faceswap)
print('unknown identities', len(ids_unknown))
print('faceswap identities', len(ids_faceswap))
print('overlap identities', len(ids_unknown & ids_faceswap))

# final interpretation
print('Interpretation: unknown spans all 500 identities and therefore looks like a base set, not a complementary FSGAN-only subset.')

# figure: distribution of unknown counts per identity
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(list(unknown_id_counts.values()), bins=10, color="#4c78a8")
ax.set_title('Unknown rows per identity')
ax.set_xlabel('Count of unknown rows per identity')
ax.set_ylabel('Number of identities')
fig_path = save(fig, 'unknown_identity_distribution')
print('saved figure', fig_path)
plt.close(fig)


unknown identity count total 500
unknown identity counts summary 1 21 8.63
unknown top 20 counts
id00076 21
id00068 14
id00183 14
id00520 14
id01075 14
id00264 14
id04073 14
id01210 14
id00569 13
id00241 13
id01052 13
id01051 13
id00032 13
id01637 13
id00841 13
id03781 13
id00100 13
id01048 13
id01182 13
id03757 13
unknown source id distinct 473
unknown source id top 20
00018 24
00241 23
00428 22
07136 21
01004 21
00272 21
00897 20
00495 20
03858 20
00358 20
06354 20
00063 19
00350 19
04701 18
02587 18
00188 18
04583 18
07108 18
00169 18
00062 18
pick identity rows 37
pick identity category/method counts
FakeVideo-FakeAudio wav2lip 12
FakeVideo-RealAudio wav2lip 10
FakeVideo-RealAudio unknown 7
FakeVideo-FakeAudio faceswap 5
RealVideo-FakeAudio none 1
RealVideo-RealAudio none 1
FakeVideo-RealAudio faceswap 1
pick identity filenames full list
00005_id05631_wavtolip.mp4 | FakeVideo-FakeAudio | wav2lip
00005_id06061_TckvmWR97vo_faceswap_id03379_wavtolip.mp4 | FakeVideo-FakeAudio | faceswa

Section 2 — method 스키마 재설계

Question: Are the video and audio axes independent and does FakeAudio imply rtvc for both FakeVideo-FakeAudio and RealVideo-FakeAudio?

In [2]:
import csv
from collections import Counter
with open(os.path.join(repo_root, 'reports', 'eda', 'method_resolved.csv'), newline='', encoding='utf-8') as f:
    resolved=list(csv.DictReader(f))

combo=Counter((r['video_swap'], r['video_lipsync'], r['audio_method']) for r in resolved)
print('top combos')
for k,v in combo.most_common(20):
    print(k, v)

by_cat_audio=Counter((r['category'], r['audio_method']) for r in resolved)
print('audio_method by category')
for (cat,am),c in sorted(by_cat_audio.items()):
    print(cat, am, c)

fakeaudio_nonrtvc=[r for r in resolved if r['category'] in {'FakeVideo-FakeAudio','RealVideo-FakeAudio'} and r['audio_method']!='rtvc']
print('FakeAudio rows with audio_method != rtvc', len(fakeaudio_nonrtvc))

fig, ax = plt.subplots(figsize=(8, 3))
combo_items = combo.most_common(7)
labels = ['|'.join(k) for k,_ in combo_items]
vals = [v for _,v in combo_items]
ax.bar(labels, vals, color="#f58518")
ax.set_title('Top combos')
ax.set_xlabel('video_swap|video_lipsync|audio_method')
ax.set_ylabel('Count')
fig_path = save(fig, 'method_combo_counts')
print('saved figure', fig_path)
plt.close(fig)


top combos
('none', 'wav2lip', 'rtvc') 9072
('none', 'wav2lip', 'none') 5015
('unresolved', 'none', 'none') 4315
('faceswap', 'wav2lip', 'rtvc') 1763
('none', 'none', 'rtvc') 500
('none', 'none', 'none') 500
('faceswap', 'none', 'none') 379
audio_method by category
FakeVideo-FakeAudio rtvc 10835
FakeVideo-RealAudio none 9709
RealVideo-FakeAudio rtvc 500
RealVideo-RealAudio none 500
FakeAudio rows with audio_method != rtvc 0
saved figure /home/scuty/project/bitamin/multimodal/reports/figures/method_combo_counts.png


Section 3 — 샘플링 계획 재산정

Question: Compare three cap options for sampling and choose the best trade-off. The real set remains fixed at 500.

In [3]:
import csv, os
from collections import Counter

with open(os.path.join(repo_root, 'reports', 'eda', 'index.csv'), newline='', encoding='utf-8') as f:
    rows=list(csv.DictReader(f))
unknown=[r for r in rows if r['method'].strip().lower()=='unknown']
faceswap=[r for r in rows if r['category']=='FakeVideo-RealAudio' and r['method'].strip().lower()=='faceswap']
wav=[r for r in rows if r['category']=='FakeVideo-RealAudio' and r['method'].strip().lower()=='wav2lip']
real=[r for r in rows if r['category']=='RealVideo-RealAudio']

cap_options=[500,1000,2000]
rows_out=[]
for include_unknown in [True, False]:
    for cap in cap_options:
        fake_groups=[faceswap, wav]
        if include_unknown:
            fake_groups.append(unknown)
        fake_total=sum(min(len(g), cap) for g in fake_groups)
        total=len(real)+fake_total
        frames=32*total
        expected_gb=round(total*0.0016,2)
        real_fake_ratio=round(len(real)/fake_total,2) if fake_total else float('inf')
        faceswap_shortfall=max(0, len(faceswap)-cap)
        rows_out.append({
            'variant':'unknown_included' if include_unknown else 'unknown_excluded',
            'cap':cap,
            'total_videos':total,
            'total_frames':frames,
            'expected_gb':expected_gb,
            'real_fake_ratio':real_fake_ratio,
            'faceswap_shortfall':faceswap_shortfall,
        })

print('sampling comparison')
for row in rows_out:
    print(row)

out_path=os.path.join(repo_root, 'reports', 'eda', 'method_sampling_comparison.csv')
with open(out_path,'w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f, fieldnames=list(rows_out[0].keys()))
    writer.writeheader()
    writer.writerows(rows_out)
print('wrote', out_path)


sampling comparison
{'variant': 'unknown_included', 'cap': 500, 'total_videos': 1879, 'total_frames': 60128, 'expected_gb': 3.01, 'real_fake_ratio': 0.36, 'faceswap_shortfall': 0}
{'variant': 'unknown_included', 'cap': 1000, 'total_videos': 2879, 'total_frames': 92128, 'expected_gb': 4.61, 'real_fake_ratio': 0.21, 'faceswap_shortfall': 0}
{'variant': 'unknown_included', 'cap': 2000, 'total_videos': 4879, 'total_frames': 156128, 'expected_gb': 7.81, 'real_fake_ratio': 0.11, 'faceswap_shortfall': 0}
{'variant': 'unknown_excluded', 'cap': 500, 'total_videos': 1379, 'total_frames': 44128, 'expected_gb': 2.21, 'real_fake_ratio': 0.57, 'faceswap_shortfall': 0}
{'variant': 'unknown_excluded', 'cap': 1000, 'total_videos': 1879, 'total_frames': 60128, 'expected_gb': 3.01, 'real_fake_ratio': 0.36, 'faceswap_shortfall': 0}
{'variant': 'unknown_excluded', 'cap': 2000, 'total_videos': 2879, 'total_frames': 92128, 'expected_gb': 4.61, 'real_fake_ratio': 0.21, 'faceswap_shortfall': 0}
wrote /home/scu

Final decisions and summary

- The identity evidence does not support the FSGAN hypothesis as a complementary subset; unknown covers all 500 identities and looks like a base set.
- The schema is still treated as unresolved for unknown rows, with `video_swap_hypothesis` reserved for later promotion.
- Sampling remains unresolved pending a user choice among the three cap options; the comparison table is saved to reports/eda/method_sampling_comparison.csv.